# Design Patterns
- Creational Patterns
- Structural Patterns
- Behavioral Patterns

## Creational Patterns
- Factory
- Singleton
- Builder

### 1. Factory Pattern
- Create objects without exposing the instantiation logic
- Systematic way to create objects

In [1]:
class Trade:
    def execute(self):
        pass

class EquityTrade(Trade):
    def execute(self):
        print("Executing equity trade")

class FXTrade(Trade):
    def execute(self):
        print("Executing FX trade")

class TradeFactory:
    @staticmethod
    def create_trade(trade_type):
        if trade_type == "equity":
            return EquityTrade()
        elif trade_type == "fx":
            return FXTrade()
        raise ValueError("Unknown trade type")

trade = TradeFactory.create_trade("equity")
trade.execute()

Executing equity trade


### 2. Singleton Pattern

- The Singleton pattern ensures that no matter how many times you try to create an object of a class, you **always get back the exact same, single instance**.
- Provides a global point of access to that instance.

In [ ]:
import threading

class MarketDataFeed:
    _instance = None
    _lock = threading.Lock()  # Avoid race conditions

    def __new__(cls):
        with cls._lock:   # block other threads
            if cls._instance is None:
                cls._instance = super(MarketDataFeed, cls).__new__(cls)
                cls._instance.connection = cls._connect()
        return cls._instance

    @staticmethod
    def _connect():
        return "Connected to market data"

feed1 = MarketDataFeed()
feed2 = MarketDataFeed()
# assert feed1 is feed2
feed1 == feed2  # True

True

- `feed1` and `feed2` are two names pointing to the same one and only `MarketDataFeed`


- `__new__`: real "constructor" method in Python
  - called before `__init__` to create the new, empty instance of the class and return it
  - `__init__`: initialize and set up the instance after it has already been created by `__new__`

    Think of it like building a house:

    - `__new__` is the contractor who **builds the physical house** and gives you the key.

    - `__init__` is the interior designer who **comes in after the house is built** to put furniture in it (e.g., self.color = "blue").

  In a Singleton, you must override `__new__` to control the creation process itself. `__init__` is too late—by the time `__init__` is called, a new object already exists. `__new__` lets you step in and say, "Wait! Instead of building a new house, just return the key to the one I already built.

- `cls`: shorthand name for "class"
  - `self` refers to a *specific instance* of a class (like `feed1`).
  
  - `cls` refers to the *class itself* (the `MarketDataFeed` blueprint).
  
  In your `__new__(cls)` method, `cls` is the `MarketDataFeed` class. This allows you to access class-level variables, like `cls._instance`, which is essential for the Singleton to work.

- `_instance = None` is the "memory" or "flag" that the Singleton logic depends on.

  - It's a class-level variable that is **initialized once when the class is defined**. It's set to None to signify that "no instance has been created yet."

    Here is the step-by-step logic:

    1. Code Starts: `MarketDataFeed._instance` is set to `None`.
   
    2. `feed1 = MarketDataFeed()` is called:

       - Python calls `__new__(cls)`.
  
        - It checks if `cls._instance` is `None`:.
        
        - This is `True` (it's None).
        
        - The if block runs:
          - `cls._instance = super(MarketDataFeed, cls).__new__(cls)`: A new instance is created and saved into `cls._instance`.

        - The method returns the new instance stored in `cls._instance`.
  
    3. `feed2 = MarketDataFeed()` is called:

       - Python calls `__new__(cls)` again.
       
       - It checks if `cls._instance` is `None`:.
       
       - This is now `False` (because `_instance` holds the object from step 2).

       - The `if` block is skipped.

       - The method immediately returns the existing instance stored in `cls._instance`.

    Without setting `_instance = None` first, the `if` check would fail on the first run, and the logic wouldn't work.

- Problems with Singleton
  - Multithreading: two threads could simultaneously check `_instance` and both find it `None`, leading to two instances being created.
    - Use **locks** or other synchronization mechanisms to ensure that only one thread can create the instance at a time.

---

## Structural Patterns
- Decorator
- Adapter
- Composite
- Proxy

### 1. Decorator

A Decorator is a function that **takes another function as an argument**, adds some new functionality to it (like "wrapping" it in extra code), and then returns the new, enhanced function.

- A decorator is a regular Python function.

- it's designed to take another function as its **only** input.

- Inside, it defines a new function (conventionally named `wrapper`).

- This `wrapper` function contains the "new" logic, and it also calls the original function.

**`@abstractmethod`**
- forces all child classes to implement a method

**`@staticmethod`**
- utility/helper functions inside classes
  - a regular function but kept inside the class for organization
- do not need access to instance (`self`) or class (`cls`) variables or methods

**`@classmethod`**
- receives the class (`cls`) instead of the instance (`self`) as the first argument
  - create objects using class-level data
  - load config
  - modify class-level state

**`@property`**
- turn methods into read-only attributes
  - computed attributes based on other instance data

In [19]:
import time
import functools

def simple_decorator(function):
    """
    A simple decorator that prints before and after the function call.
    This will ONLY work for functions that take no arguments.
    """
    def wrapper():
        print("Simple Decorator:")
        print("--- Before the function ---")
        result = function()     # Use result to store return value
        print(f"Result was: {result}")
        print("--- After the function ---")
        return result
    return wrapper

@simple_decorator
def say_hello():
    """Returns a simple greeting."""
    print("Running say_hello()")
    return "hello"

print("--- Calling say_hello ---")
say_hello()

--- Calling say_hello ---
Simple Decorator:
--- Before the function ---
Running say_hello()
Result was: hello
--- After the function ---


'hello'

In [20]:
def advanced_decorator(function):
    """
    A robust decorator that times the function, logs its name and arguments.
    It uses *args and **kwargs to work with ANY function.
    """
    # functools.wraps preserves the original function's name and docstring
    @functools.wraps(function) 
    def wrapper(*args, **kwargs):
        print(f"\nCalling function: {function.__name__} ---")
        print(f"Positional Args: {args}")
        print(f"Keyword Args: {kwargs}")
        
        start = time.time()
        result = function(*args, **kwargs) # Pass arguments to the function
        end = time.time()
        
        print(f"Result: {result}")
        print(f"Execution Time: {end - start:.6f} seconds")
        print(f"Finished: {function.__name__} ---")
        return result
    return wrapper

@advanced_decorator
def say_goodbye(name, greeting="Goodbye"):
    """Returns a named greeting."""
    print("Running say_goodbye()")
    time.sleep(0.1) # Simulate work
    return f"{greeting}, {name}!"

print("\n--- Calling say_goodbye (positional) ---")
say_goodbye("Shen-Ching")

print("\n--- Calling say_goodbye (keyword) ---")
say_goodbye(name="Seb", greeting="Adios")


--- Calling say_goodbye (positional) ---

Calling function: say_goodbye ---
Positional Args: ('Shen-Ching',)
Keyword Args: {}
Running say_goodbye()
Result: Goodbye, Shen-Ching!
Execution Time: 0.105045 seconds
Finished: say_goodbye ---

--- Calling say_goodbye (keyword) ---

Calling function: say_goodbye ---
Positional Args: ()
Keyword Args: {'name': 'Seb', 'greeting': 'Adios'}
Running say_goodbye()
Result: Adios, Seb!
Execution Time: 0.105046 seconds
Finished: say_goodbye ---


'Adios, Seb!'

In [21]:
def audit(function):
    """
    A decorator for a specific purpose: auditing.
    """
    @functools.wraps(function)
    def wrapper(*args, **kwargs):
        print(f"AUDIT: Running {function.__name__}...")
        result = function(*args, **kwargs)
        print(f"AUDIT: {function.__name__} finished.")
        return result
    return wrapper

@audit
@advanced_decorator # You can stack decorators!
def execute_trade(trade_type, quantity):
    """Executes a trade."""
    print(f"Running execute_trade(): Executing {quantity} of {trade_type}")
    return f"Executed {trade_type} trade"

print("\n\n--- Calling execute_trade (stacked) ---")
execute_trade("AAPL", 100)



--- Calling execute_trade (stacked) ---
AUDIT: Running execute_trade...

Calling function: execute_trade ---
Positional Args: ('AAPL', 100)
Keyword Args: {}
Running execute_trade(): Executing 100 of AAPL
Result: Executed AAPL trade
Execution Time: 0.000014 seconds
Finished: execute_trade ---
AUDIT: execute_trade finished.


'Executed AAPL trade'

- Use `@functools.wraps(function)` to preserve the original function's metadata (like its name and docstring).

In [22]:
print(f"\nFunction name of say_goodbye: {say_goodbye.__name__}")


Function name of say_goodbye: say_goodbye


In [23]:
# Without functools.wraps

def deco(fn):
    def wrapper():
        return fn()
    return wrapper

@deco
def hello():
    "Says hello"
    return "hi"

print(hello.__name__)  # wrapper  ❌
print(hello.__doc__)   # None ❌

wrapper
None


In [24]:
# With functools.wraps
def deco(fn):
    @functools.wraps(fn)
    def wrapper():
        return fn()
    return wrapper

@deco
def hello():
    "Says hello"
    return "hi"

print(hello.__name__)  # hello   ✔
print(hello.__doc__)   # "Says hello" ✔


hello
Says hello


### *args & **kwargs
***args**:
- Arbitrary number of positional arguments
- Tuple
  - Order matters
  - No names, just values

****kwargs**:
- Arbitrary number of keyword arguments
- Dictionary
  - Order does not matter
  - Names and values

In [25]:
def demo(*args, **kwargs):
    print("Positional:", args)
    print("Keyword:", kwargs)

demo(1, 2, name="Seb", school="UChicago")

Positional: (1, 2)
Keyword: {'name': 'Seb', 'school': 'UChicago'}


---

## Behavioral Patterns
- Observer
- Iterator
  - Generator
- Strategy
- Command

### 1. Observer Pattern

- The Observer Pattern is a design pattern used to create a **one-to-many relationship between objects**.
  - Position Manager vs Trading Strategy
  - Exchange vs Trading System

- The Subject (`PositionManager`)

  - This is the object that has the **important data or state**.

  - It's the "thing being watched."

  - Its job is to **maintain a list of its `observers`** and **notify them** when its state changes.

- The Observer (`TradingStrategy`)

  - This is an object that **wants to know when the Subject's state changes**.

  - It "subscribes" to the Subject and provides an `update()` method that the `Subject` can call.

In [ ]:
class PositionManager:  # This is the SUBJECT
    def __init__(self) -> None:
        self.observer = list()
        self.position = 0
    
    def update_trade(self, side, price, quantity):
        if side == "buy":
            p = price * quantity
        else:
            p = -price * quantity
        self.position += p
        self.notifyObserver(self.position)
    
    def registerObserver(self, trading_strategy):
        # Adds an observer to the "subscription list"
        self.observer.append(trading_strategy)
    
    def notifyObserver(self, position):
        # "Publishes" the update to all subscribers
        for ts in self.observer:
            ts.update(position)

class TradingStrategy:  # This is the OBSERVER
    def __init__(self) -> None:
        self.position = 0
    
    # This is the "contract" method the Subject needs
    def update(self, position):
        self.position = position

pm1 = PositionManager()
ts1 = TradingStrategy()
ts2 = TradingStrategy()
ts3 = TradingStrategy()

# The Observers "subscribe" to the Subject
pm1.registerObserver(ts1)
pm1.registerObserver(ts2)
pm1.registerObserver(ts3)

# The Subject's state changes, and it notifies all observers
pm1.update_trade("buy", 10, 100)

for ts in [ts1, ts2, ts3]:
    print(ts.position)

1000
1000
1000


1. Flexibility:

   - The `PositionManager` (Subject) doesn't know or care that its observers are `TradingStrategy` objects.
   
   - They could be a `RiskManager`, a `DashboardUI`, or a `Logger`.
   
   - All it knows is that they have an `update(position)` method it can call.

2. Scalability:

   - You can easily add a new observer (`ts4`, `ts5`) by just calling `pm1.registerObserver()`.

   - You don't have to change any code inside the `PositionManager` class.

3. One-to-Many Updates:

   - When the position changes (one event), all interested objects (`ts1`, `ts2`, `ts3`) are updated **automatically**.

### 2 Iterator Pattern
an object that actually produces values one by one.

**`__iter__`**:
  - Returns the iterator object itself.
  - Called when an iterator is created.

**`__next__`**:
  - Returns the next value from the iterator.
  - Raises `StopIteration` when there are no more values to return.

In [ ]:
for x in iterable:
    pass

# EQUAL TO
iterator = iter(iterable)   # calls iterable.__iter__()
while True:
    try:
        x = next(iterator)  # calls iterator.__next__()
    except StopIteration:
        break
    pass

In [34]:
class TradeBook:
    def __init__(self, trades):
        # store an iterator so __next__ can call next(self.trades)
        self.trades = iter(trades)

    def __iter__(self):
        # object is its own iterator
        return self

    def __next__(self):
        return next(self.trades)

book = TradeBook(["T1", "T2", "T3"])
print(next(book))  # T1
print(next(book))  # T2
print(next(book))  # T3

T1
T2
T3


In [ ]:
## BST
class BinarySearchTree:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

    # This __iter__ uses 'yield' to become a generator (an easy way to make an iterator)
    def __iter__(self):
        # 1. Yield all items from the left child
        if self.left:
            # pause this function
            # and go run the entire __iter__ method for my left child (self.left)
            # Hand back all the values from that left child one by one
            yield from self.left
        
        # 2. Yield the current node's value back to the for loop
        yield self.value
        
        # 3. Yield all items from the right child
        if self.right:
            yield from self.right

# --- Usage ---
#          10
#         /  \
#        5    15
tree = BinarySearchTree(10, BinarySearchTree(5), BinarySearchTree(15))  

# The 'for' loop just works, hiding all the tree logic!
print("In-order traversal:")
for val in tree:
    print(val)

## Infinite Sequences
class CountUp:
    def __init__(self, start=0):
        self.current = start

    def __iter__(self):
        # The object is its own iterator
        # called once at the start of the loop (say: CountUp itself is the iterator)
        return self

    def __next__(self):
        # Python knows this is the method to get the next item
        # This is the method the 'for' loop calls to get the *next* item
        # called repeatedly by the 'for' loop until StopIteration is raised
        num = self.current
        self.current += 1
        return num

counter = CountUp(1)

for i in counter:
    print(i)
    if i == 3:
        break  # Must have a break, or it loops forever!

In-order traversal:
5
10
15
1
2
3


- `__iter__` is the special method Python automatically calls when you try to loop over an object.

  - return a iterator

    - `iter(self.trades)` in `TradeBook`
    
    - `yield` in `BinarySearchTree`
    
    - `CountUp` (and it's `__next__`) in `CountUp`

### 2-1. Generator Pattern
- lazy evaluation
- extremely memory efficient
- `yield`
  - returns a value
  - pauses the function
  - remembers where it left off
  - A function with `yield` becomes a generator
- built-in iterator (Generator is an Iterator)
  - `__iter__()`
  - `__next__()`
  - raises `StopIteration` when done

1. `yield x`
   - Output the value `x` and pause the function's state
2. `next()`
   - Resumes the function from where it left off
3. raise `StopIteration` when done
   - Automatically raised when the generator is exhausted

| yield                                    | return                                      |
|-------------------------------------------|----------------------------------------------|
| returns **one value at a time**           | returns one value **and ends function**      |
| function **pauses**                       | function **terminates**                      |
| creates a **generator**                   | creates **normal function**                  |
| **maintains internal state**              | **does not** keep state                      |


In [30]:
def trade_stream():
    for i in range(1, 5):
        yield f"Trade-{i}"

ts = trade_stream()
print(next(ts))  # Trade-1
print(next(ts))  # Trade-2
print(next(ts))  # Trade-3
print(next(ts))  # Trade-4

Trade-1
Trade-2
Trade-3
Trade-4


- Instead of creating a list `["Trade-1", "Trade-2", "Trade-3"]`. It creates a generator object.

  1. The `for` loop asks for the first item.

  2. `trade_stream` runs, `i` becomes 1, and it yields "Trade-1".

  3. `trade_stream` pauses.

  4. The for loop prints "Trade-1".

  5. The for loop asks for the next item.

  6. `trade_stream` resumes, `i` becomes 2, and it yields "Trade-2".

  7. ...and so on.

- Why this matters: What if your function was `range(1, 1000000000)`?

  - A normal function that builds a list would crash your computer by trying to create a list with one billion items in memory.

  - A generator function (like `trade_stream`) uses almost no memory. It only ever holds the one current value (`f"Trade-{i}"`) at any given time.

- This "one at a time" processing is what makes it ideal for streaming large datasets, like reading a 100GB log file one line at a time. You process one line, then the generator yields the next line, without ever loading the whole file into memory.